# 第7章 分区运营预测与典型服务日

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch07-spatiotemporal-v1`  
**必做：** 历史同期均值；单区域AR；多区域VAR；时空画像KMeans及ARI  
**对象：** 四区域保留上车记录：条/小时；日型为168维占比  
**样本：** 预测四区域×7天×24小时=672；聚类31日×7类地区×24小时  
**划分：** 1-21日训练，22-24日验证选p，25-31日测试；全月聚类与预测严格分开  
**比较：** AR/VAR都从p=1/3/24按验证RMSE选；K=2/3/4/5与seed=42/7/19；不把全月聚类喂给留出预测

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/learning-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 分开建立预测矩阵与探索性日画像
四区域预测有时间留出；全月聚类只用于探索，不可将其作为留出预测的输入。


In [ ]:
FILES=['projects/data/taxi.json']
raw=json.loads((ROOT/FILES[0]).read_text(encoding='utf-8'))['rows']
NODES=['Manhattan','Brooklyn','Queens','Bronx']
boroughs=sorted({r[1] for r in raw})
matrix=np.zeros((31*24,4));profiles=np.zeros((31,len(boroughs)*24))
for date,borough,hour,count in raw:
    day=int(date[-2:])-1
    profiles[day,boroughs.index(borough)*24+hour]+=count
    if borough in NODES:matrix[day*24+hour,NODES.index(borough)]+=count
training=matrix[:21*24]
valid_idx=np.arange(21*24,24*24);test_idx=np.arange(24*24,31*24)
actual=matrix[test_idx]
print(matrix.shape,profiles.shape,actual.shape)


## 2. 构造滞后特征并在验证期选阶
查看X、y和模型系数。单区域AR只读取该区域自己的滞后，VAR读取全部四区域。


In [ ]:
from learning_support import lag_design,fit_lag
LAGS=[1,3,24]
X_example,y_example=lag_design(training,3)
print('VAR lag-3 design:',X_example.shape,y_example.shape)
predictions={};selected={};validation=[]
for name,multi in [('AR',False),('VAR',True)]:
    trials=[]
    for lag in LAGS:
        model,predict=fit_lag(training,lag,multi)
        pred=np.maximum(0,np.array([predict(matrix[:t]) for t in valid_idx]))
        score=metrics(matrix[valid_idx].ravel(),pred.ravel())['RMSE']
        validation.append([name,lag,score]);trials.append((score,lag,model,predict))
    _,lag,model,predict=min(trials,key=lambda r:(r[0],r[1]))
    selected[name]=lag
    predictions[name]=np.maximum(0,np.array([predict(matrix[:t]) for t in test_idx]))
predictions['Calendar']=np.array([training[t%168::168].mean(axis=0) for t in test_idx])
predictions['Last']=matrix[test_idx-1]
display(pd.DataFrame(validation,columns=['model','lag','validation_RMSE']))


## 3. 比较共同672个区域小时
请解释：总体RMSE改善是否意味着每个区域都改善？


In [ ]:
scores=metric_table(actual.ravel(),{k:v.ravel() for k,v in predictions.items()});display(scores)
node_scores=pd.DataFrame([{'region':region,'model':name,**metrics(actual[:,j],p[:,j])} for j,region in enumerate(NODES) for name,p in predictions.items()])
display(node_scores)
REGION='Queens'
j=NODES.index(REGION)
plt.figure(figsize=(11,4));plt.plot(actual[:,j],label='Observed')
for name in ['AR','VAR','Calendar']:plt.plot(predictions[name][:,j],label=name)
plt.xlabel('Test-hour index, Jan 25-31');plt.ylabel('Retained pickups/hour');plt.title(REGION);plt.legend();save_plot(ROOT,7,'regional_prediction')


## 4. 独立实施12组KMeans与成员一致性核查
中心向量表示相对时空形态。不要将簇编号直接命名为已确认的出行目的。


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score,adjusted_rand_score
K_VALUES=[2,3,4,5]
SEEDS=[42,7,19]
shares=profiles/profiles.sum(axis=1,keepdims=True)
memberships={};centers={};cluster_rows=[]
for k in K_VALUES:
    reference=None
    for seed in SEEDS:
        model=KMeans(n_clusters=k,n_init=10,random_state=seed).fit(shares)
        if reference is None:reference=model.labels_
        memberships[f'{k}-{seed}']=model.labels_.tolist()
        centers[f'{k}-{seed}']=model.cluster_centers_.tolist()
        cluster_rows.append([k,seed,float(silhouette_score(shares,model.labels_)),float(adjusted_rand_score(reference,model.labels_)),int(np.bincount(model.labels_).min())])
cluster_scores=pd.DataFrame(cluster_rows,columns=['K','seed','silhouette','ARI_vs_first_seed','smallest_cluster_days']);display(cluster_scores)


## 5. 联读形态与原始规模
修改VIEW，看看同一日期是否换组。柱高是日总量，颜色是占比聚类，两者不是同一变量。


In [ ]:
VIEW='4-42'
plt.figure(figsize=(11,4));plt.bar(np.arange(1,32),profiles.sum(axis=1),color=plt.get_cmap('tab10')(np.array(memberships[VIEW])))
plt.xlabel('January day');plt.ylabel('Retained pickups/day');plt.title(VIEW);save_plot(ROOT,7,'typical_days')
ids=[f'2024-01-{t//24+1:02}T{t%24:02}:00|{b}' for t in test_idx for b in NODES]
outputs={'predictions':predictions_table(ids,{k:v.ravel() for k,v in predictions.items()}),'memberships':memberships,'centers':centers,'cluster_scores':cluster_scores.to_dict('records')}
primary(ROOT,7,FILES,{'nodes':NODES,'lags':LAGS,'selected_lags':selected,'K':K_VALUES,'seeds':SEEDS},outputs)
csv_file(ROOT/'outputs/ch07/predictions.csv',outputs['predictions']['columns'],outputs['predictions']['rows'])
csv_file(ROOT/'outputs/ch07/typical_day_memberships.csv',['date',*memberships],[[f'2024-01-{i+1:02}',*[v[i] for v in memberships.values()]] for i in range(31)])
report(ROOT,7,'分区运营研判与典型服务日报告',{'总体误差':scores.to_string(index=False),'分区误差':node_scores.to_string(index=False),'聚类稳定性':cluster_scores.to_string(index=False)},['哪些区域适合采用多区域预测？','哪些日型对初始化敏感？','运营假设还需哪些供给或候车数据验证？'])
